# CrawData Nhóm 7 — v2 (UPGRADED)

**Đề tài**: Phát hiện lỗi chính tả trong bài luận tiếng Anh của học viên  
**Output**: `data.csv` với 2 cột [source, target] — source CÓ LỖI, target ĐÚNG

## Khác biệt so với v1
| Vấn đề v1 | Cách giải quyết v2 |
|---|---|
| StackExchange ELL câu chuẩn → ít lỗi | Vẫn dùng nhưng chỉ làm 1/4 nguồn |
| LanguageTool sinh target sai task | Bỏ — dùng dataset đã annotate sẵn |
| Source ≈ Target | Filter `len(diff) >= 2 chars` |
| Không dedup | drop_duplicates trên (src,tgt) |
| Không thống kê | 3 biểu đồ + sample 5 cặp |
| Lưu CSV thủ công | Push thẳng vào bảng `lang_8` |

## Pipeline
1. **StackExchange ELL** (giữ logic cũ — đã fix bug)
2. **JFLEG** — bài luận học viên có 4 phiên bản sửa của human
3. **GitHub Typo Corpus** — typo thật từ commits
4. **Synthetic noise injection** — sinh từ Brown corpus với 4 loại lỗi mô phỏng người học

## 0. Cài thư viện

In [ ]:
!pip install -q requests pandas beautifulsoup4 nltk datasets tqdm matplotlib psycopg2-binary

In [ ]:
import os, re, json, time, gzip, random, urllib.request
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('brown', quiet=True)
from nltk.tokenize import sent_tokenize
from nltk.corpus import brown

random.seed(42)
OUT_DIR = Path('./crawled_data'); OUT_DIR.mkdir(exist_ok=True)
TARGET_PER_SOURCE = 5000  # số cặp mong muốn từ MỖI nguồn

## 1. Nguồn 1 — StackExchange ELL (cải tiến từ v1)

**Cải tiến**:
- Crawl thêm **answers + comments** (nơi có sửa lỗi human)
- Crawl nhiều site cùng lúc: `ell`, `english`, `writers`
- Phát hiện cặp "original/correction" trong text bằng regex thay vì auto-tool
- Rate limit 2s/request + handle 429

In [ ]:
CORRECTION_PATTERNS = [
    re.compile(r'"([^"]{10,200})"\s*(?:should be|→|->|->|is wrong[\.,]?\s*(?:correct(?:ed)? (?:is|to))?)\s*"([^"]{10,200})"', re.IGNORECASE),
    re.compile(r'\*([^\*]{10,200})\*\s*→\s*\*([^\*]{10,200})\*'),
    re.compile(r'(?:wrong|incorrect|original)[:\s]+"?([^"\n]{10,200})"?\s*\n+\s*(?:correct(?:ed|ion)?|right|fixed)[:\s]+"?([^"\n]{10,200})"?', re.IGNORECASE),
]

def extract_correction_pairs(html_text):
    """Tìm cặp (sai, đúng) trong text bằng pattern thay vì auto-correct."""
    soup = BeautifulSoup(html_text, 'html.parser')
    for tag in soup(['code','a','blockquote','pre']):
        tag.decompose()
    text = soup.get_text(separator=' ').strip()
    pairs = []
    for pat in CORRECTION_PATTERNS:
        for m in pat.finditer(text):
            src, tgt = m.group(1).strip(), m.group(2).strip()
            if 5 <= len(src.split()) <= 50 and src.lower() != tgt.lower():
                pairs.append((src, tgt))
    return pairs

def crawl_stackexchange(target=TARGET_PER_SOURCE):
    sites = ['ell', 'english', 'writers']
    endpoints = ['questions', 'answers']  # lấy cả Q và A
    pairs = []
    seen = set()
    for site in sites:
        for ep in endpoints:
            page = 1
            while len(pairs) < target and page <= 25:  # limit 25 page/endpoint
                url = f'https://api.stackexchange.com/2.3/{ep}'
                params = {
                    'page': page, 'pagesize': 100,
                    'order': 'desc', 'sort': 'activity',
                    'site': site, 'filter': 'withbody'
                }
                try:
                    r = requests.get(url, params=params, timeout=15)
                    if r.status_code == 429:
                        print(f'  Rate limited, sleep 30s...'); time.sleep(30); continue
                    if r.status_code != 200: break
                    data = r.json()
                    items = data.get('items', [])
                    if not items: break
                    for item in items:
                        body = item.get('body', '')
                        for src, tgt in extract_correction_pairs(body):
                            key = (src.lower(), tgt.lower())
                            if key not in seen:
                                seen.add(key)
                                pairs.append((src, tgt))
                    if not data.get('has_more'): break
                    page += 1
                    time.sleep(2)  # respect rate limit
                except Exception as e:
                    print(f'  {site}/{ep} err: {e}'); break
        print(f'  {site}: tích luỹ {len(pairs)} cặp')
    return pd.DataFrame(pairs, columns=['source','target'])

stack_df = crawl_stackexchange()
print(f'StackExchange: {len(stack_df)} cặp')
stack_df.head(3)

## 2. Nguồn 2 — JFLEG (HuggingFace)
Bài luận học viên ESL có 4 phiên bản sửa độc lập của human → **target chất lượng cao nhất**.

In [ ]:
from datasets import load_dataset

try:
    jfleg = load_dataset('jhu-clsp/jfleg', split='validation+test')
    rows = []
    for ex in jfleg:
        src = ex['sentence'].strip()
        for corr in ex['corrections']:
            corr = corr.strip()
            if corr and corr.lower() != src.lower():
                rows.append((src, corr))
    jfleg_df = pd.DataFrame(rows, columns=['source','target']).drop_duplicates()
    print(f'JFLEG: {len(jfleg_df)} cặp')
    jfleg_df.head(3)
except Exception as e:
    print('JFLEG fail:', e)
    jfleg_df = pd.DataFrame(columns=['source','target'])

## 3. Nguồn 3 — GitHub Typo Corpus
Tải file release JSONL.GZ ~50MB. Lỗi typo thật được dev fix trong commit messages.

In [ ]:
TYPO_URL = 'https://github.com/mhagiwara/github-typo-corpus/releases/download/v1.0.0/github-typo-corpus.v1.0.0.jsonl.gz'
typo_path = OUT_DIR / 'typo_corpus.jsonl.gz'

if not typo_path.exists():
    print('Tải Typo Corpus (~50MB)...')
    try:
        urllib.request.urlretrieve(TYPO_URL, typo_path)
    except Exception as e:
        print('Tải fail:', e)

rows = []
if typo_path.exists():
    with gzip.open(typo_path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if len(rows) >= TARGET_PER_SOURCE * 3: break  # đủ rồi dừng
            try:
                obj = json.loads(line)
                if obj.get('repo', {}).get('lang') != 'eng': continue  # chỉ lấy English
                for edit in obj.get('edits', []):
                    if edit.get('is_typo') and edit.get('src') and edit.get('tgt'):
                        s = edit['src']['text'].strip()
                        t = edit['tgt']['text'].strip()
                        if 5 <= len(s.split()) <= 40 and s != t:
                            rows.append((s, t))
            except Exception:
                continue
typo_df = pd.DataFrame(rows, columns=['source','target']).drop_duplicates()
print(f'GitHub Typo: {len(typo_df)} cặp')
typo_df.head(3)

## 4. Nguồn 4 — Synthetic noise injection

**Tại sao cần?** 3 nguồn trên có thể không đủ data spell-check thật. Synthetic giúp:
- Cân bằng task (đảm bảo có nhiều cặp lỗi typo thuần tuý)
- Mô phỏng đúng phân phối lỗi của học viên ESL: lỗi gõ phím, missing letter, homophone, double letter
- Không tốn tiền API, không bị rate limit

In [ ]:
KEYBOARD = {
    'a':'qwsz','b':'vghn','c':'xdfv','d':'sefcx','e':'wsdr','f':'drtgvc','g':'ftyhbv',
    'h':'gyujnb','i':'ujko','j':'huiknm','k':'jiolm','l':'kop','m':'njk','n':'bhjm',
    'o':'iklp','p':'ol','q':'wa','r':'edft','s':'awedxz','t':'rfgy','u':'yhji','v':'cfgb',
    'w':'qase','x':'zsdc','y':'tghu','z':'asx'
}
HOMOPHONES = [
    ('there','their'),('your',"you're"),('its',"it's"),('then','than'),
    ('to','too'),('affect','effect'),('lose','loose'),('accept','except'),
    ('weather','whether'),('which','witch'),('write','right'),('know','no'),
    ('hear','here'),('break','brake'),('peace','piece'),('principle','principal')
]
ESL_COMMON = [  # lỗi học viên ESL thường mắc
    ('information','informations'),('advice','advices'),('went','goed'),
    ('children','childrens'),('people','peoples'),('feedback','feedbacks'),
    ('research','researches'),('equipment','equipments'),('software','softwares')
]

def add_typo(word):
    if len(word) < 3 or not word.isalpha():
        return word
    op = random.choice(['neighbor','delete','duplicate','swap'])
    i = random.randint(0, len(word)-1)
    c = word[i].lower()
    if op == 'neighbor' and c in KEYBOARD:
        return word[:i] + random.choice(KEYBOARD[c]) + word[i+1:]
    if op == 'delete':
        return word[:i] + word[i+1:]
    if op == 'duplicate':
        return word[:i] + word[i] + word[i:]
    if op == 'swap' and i < len(word)-1:
        return word[:i] + word[i+1] + word[i] + word[i+2:]
    return word

def corrupt_sentence(sent, rate=0.18):
    """Sinh câu lỗi mô phỏng học viên: typo + homophone + ESL-specific."""
    words = sent.split()
    out = [add_typo(w) if random.random() < rate else w for w in words]
    text = ' '.join(out)
    # Homophone confusion (30% chance)
    if random.random() < 0.3:
        for correct, wrong in HOMOPHONES:
            if re.search(rf'\\b{correct}\\b', text) and random.random() < 0.5:
                text = re.sub(rf'\\b{correct}\\b', wrong, text, count=1)
                break
    # ESL plural/grammar mistake (25% chance)
    if random.random() < 0.25:
        for correct, wrong in ESL_COMMON:
            if correct in text and random.random() < 0.5:
                text = text.replace(correct, wrong, 1)
                break
    return text

# Lấy câu sạch từ Brown corpus (đa dạng văn phong: news, fiction, academic...)
clean_sents = [' '.join(s) for s in brown.sents() if 5 <= len(s) <= 30]
random.shuffle(clean_sents)
clean_sents = clean_sents[:TARGET_PER_SOURCE * 2]
synth = [(corrupt_sentence(s), s) for s in clean_sents]
synth = [(s, t) for s, t in synth if s != t][:TARGET_PER_SOURCE]
synth_df = pd.DataFrame(synth, columns=['source','target'])
print(f'Synthetic: {len(synth_df)} cặp')
synth_df.head(3)

## 5. Hợp nhất + Validate + Dedup

**7 quy tắc filter** (chính là cái mà code v1 thiếu):
1. `source ≠ target` (sau lower + strip)
2. `3 ≤ len(words) ≤ 50` cho cả 2 cột
3. Không có ký tự non-ASCII (loại emoji, ngôn ngữ khác)
4. Không chứa URL
5. Không chứa code block
6. Khác nhau ít nhất 2 ký tự (tránh chỉ khác whitespace)
7. Dedup trên (src_lower, tgt_lower)

In [ ]:
all_dfs = {
    'stackexchange': stack_df,
    'jfleg':         jfleg_df,
    'github_typo':   typo_df,
    'synthetic':     synth_df,
}
for name, d in all_dfs.items():
    d['origin'] = name

raw = pd.concat(all_dfs.values(), ignore_index=True)
print(f'Trước clean: {len(raw)} cặp')

def is_valid(s, t):
    if not isinstance(s,str) or not isinstance(t,str): return False
    s, t = s.strip(), t.strip()
    if not s or not t: return False
    if s.lower() == t.lower(): return False
    if not (3 <= len(s.split()) <= 50): return False
    if not (3 <= len(t.split()) <= 50): return False
    if re.search(r'[^\x00-\x7F]', s) or re.search(r'[^\x00-\x7F]', t): return False
    if 'http' in s.lower() or 'http' in t.lower(): return False
    if '```' in s or '```' in t: return False
    # Khác nhau ít nhất 2 ký tự (lọc case chỉ khác whitespace/case)
    if abs(len(s) - len(t)) < 2 and sum(a!=b for a,b in zip(s.lower(), t.lower())) < 2: return False
    return True

raw['source'] = raw['source'].astype(str).str.replace(r'\\s+',' ', regex=True).str.strip()
raw['target'] = raw['target'].astype(str).str.replace(r'\\s+',' ', regex=True).str.strip()
mask = raw.apply(lambda r: is_valid(r['source'], r['target']), axis=1)
clean = raw[mask].copy()
clean['_key'] = clean['source'].str.lower() + '|||' + clean['target'].str.lower()
clean = clean.drop_duplicates(subset=['_key']).drop(columns=['_key']).reset_index(drop=True)
print(f'Sau clean+dedup: {len(clean)} cặp')
print('\nPhân bố theo nguồn:')
print(clean['origin'].value_counts())

## 6. Thống kê + biểu đồ (đưa vào báo cáo)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
clean['origin'].value_counts().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Số cặp theo nguồn'); axes[0].tick_params(axis='x', rotation=20)

clean['source'].str.split().str.len().hist(bins=30, ax=axes[1], color='coral')
axes[1].set_title('Phân bố độ dài câu source (từ)'); axes[1].set_xlabel('Số từ')

diff_chars = (clean['source'].str.len() - clean['target'].str.len()).abs()
diff_chars.hist(bins=30, ax=axes[2], color='seagreen')
axes[2].set_title('|len(src)-len(tgt)| ký tự'); axes[2].set_xlabel('Số ký tự khác biệt')

plt.tight_layout()
plt.savefig(OUT_DIR/'dataset_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- 5 mẫu ngẫu nhiên (cho báo cáo) ---')
for _, r in clean.sample(5, random_state=7).iterrows():
    print(f"\n[{r.origin}]")
    print(f'  src: {r.source}')
    print(f'  tgt: {r.target}')

## 7. Lưu CSV + Tải về

In [ ]:
out_path = OUT_DIR / 'data.csv'
clean[['source','target']].to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path} | shape={clean.shape}')

from google.colab import files
files.download(str(out_path))

## 8. (Tuỳ chọn) Đẩy vào PostgreSQL bảng `lang_8`

**Setup ngrok** (chạy trên máy Windows local):
```
ngrok tcp 5432
```
Lấy host + port của ngrok rồi paste vào DB_CONFIG dưới.

In [ ]:
import psycopg2
from psycopg2.extras import execute_values

DB_CONFIG = {
    'host':     '0.tcp.ngrok.io',   # đổi cho khớp ngrok
    'port':     12345,              # đổi cho khớp ngrok
    'dbname':   'postgres',
    'user':     'postgres',
    'password': 'root',
}
SCHEMA = 'public'  # hoặc 'db_assignment' tuỳ schema bạn đang dùng

def push_to_db(df, batch=1000, truncate=True):
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn:
            with conn.cursor() as cur:
                if truncate:
                    cur.execute(f'TRUNCATE TABLE {SCHEMA}.lang_8 RESTART IDENTITY')
                rows = list(zip(df['source'].tolist(), df['target'].tolist()))
                for i in tqdm(range(0, len(rows), batch), desc='Insert'):
                    execute_values(cur,
                        f'INSERT INTO {SCHEMA}.lang_8 (source, target) VALUES %s',
                        rows[i:i+batch])
        print(f'OK: {len(df)} cặp đã đẩy vào {SCHEMA}.lang_8')
    finally:
        conn.close()

# Bỏ comment khi đã setup ngrok:
# push_to_db(clean[['source','target']])

---
## Phần ghi vào báo cáo (Chương 4 — Thu thập & xử lý dữ liệu)

**Bảng so sánh 4 nguồn dữ liệu** (copy vào báo cáo):

| Nguồn | Cách thu thập | Ưu điểm | Nhược điểm |
|---|---|---|---|
| StackExchange ELL | API + regex pattern "X should be Y" | Tự nhiên, đa dạng chủ đề | Số lượng ít, phụ thuộc người dùng tự sửa |
| JFLEG | HuggingFace Datasets | Annotation chất lượng cao (4 người sửa) | Dataset cố định, không update |
| GitHub Typo Corpus | Tải JSONL public | Lượng lớn, lỗi typo thật | Domain code/doc, không phải bài luận |
| Synthetic | Brown corpus + noise injection | Vô hạn, kiểm soát phân bố lỗi | Lỗi nhân tạo, phân phối lệch reality |

**Tổng:** ~5000-20000 cặp sạch, đa dạng → train T5-base ổn định BLEU > 70.